# Feature Engineering — Previsão de Churn

## Objetivo

Nesta etapa vamos transformar os dados brutos das sessões em informações mais úteis para o modelo de Machine Learning.

Atualmente:
- cada linha representa uma sessão de jogo.

Mas o modelo precisa entender:
- o comportamento geral do usuário.

Por isso vamos criar métricas como:
- quantidade de sessões,
- valor médio apostado,
- tempo desde a última sessão,
- uso de bônus,
- nível de engajamento.

Essas novas colunas são chamadas de **features**.

# Por que isso é importante?

Os dados brutos sozinhos normalmente não são suficientes para prever churn.

Exemplo:
- uma aposta isolada não diz muita coisa,
- mas a média de apostas do usuário pode mostrar queda de interesse.

O objetivo da Feature Engineering é transformar comportamento em números que o modelo consiga entender.

# Importação das bibliotecas

Nesta etapa importamos:
- funções do PySpark,
- biblioteca de logging.

O logging ajuda a acompanhar a execução do notebook e é considerado uma boa prática em projetos.

In [0]:
from pyspark.sql.functions import *
import logging


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("feature_engineering")
logger.info("Iniciando processo de feature engineering...")

In [0]:
try:

    df = spark.table(
        "fiap_analytics.bronze.rw_db_dataset_churn"
    )

    logger.info("Dataset carregado com sucesso.")

except Exception as e:

    logger.error(f"Erro ao carregar dataset: {e}")

    raise

In [0]:
df = df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"))
)

df = df.withColumn(
    "date",
    to_date(col("date"))
)

logger.info("Colunas de data convertidas.")

In [0]:
df = df.fillna({
    "deposit_amount": 0,
    "withdrawal_amount": 0,
    "sport_type": "Unknown"
})

logger.info("Valores nulos tratados.")

In [0]:
sessions_df = df.groupBy("user_id").agg(
    count("session_id").alias("total_sessions")
)

logger.info("Feature total_sessions criada.")

In [0]:
bet_df = df.groupBy("user_id").agg(
    avg("bet_amount").alias("avg_bet_amount")
)

logger.info("Feature avg_bet_amount criada.")

In [0]:
deposit_df = df.groupBy("user_id").agg(
    sum("deposit_amount").alias("total_deposit_amount")
)

logger.info("Feature total_deposit_amount criada.")

In [0]:
bonus_df = df.groupBy("user_id").agg(
    avg(col("bonus_used").cast("int")).alias("bonus_usage_rate")
)

logger.info("Feature bonus_usage_rate criada.")

In [0]:
engagement_df = df.groupBy("user_id").agg(
    
    avg("session_length_minutes").alias("avg_session_length"),
    
    avg("games_played").alias("avg_games_played")
)

logger.info("Features de engajamento criadas.")

## Feature: Recência

Esta é uma das features mais importantes em churn.

Ela calcula:
- quantos dias passaram desde a última sessão do usuário.

Quanto maior esse valor:
- maior o risco de churn.

In [0]:
recency_df = df.groupBy("user_id").agg(
    max("date").alias("last_session_date")
)

recency_df = recency_df.withColumn(
    "days_since_last_session",
    
    datediff(current_date(), col("last_session_date"))
)

logger.info("Feature de recência criada.")

## Feature: Atividade no Final de Semana

Esta feature mede:
- quanto o usuário joga nos finais de semana.

Alguns usuários possuem comportamento muito mais ativo nesses períodos.

In [0]:
weekend_df = df.groupBy("user_id").agg(
    avg(col("is_weekend").cast("int")).alias("weekend_activity_ratio")
)

logger.info("Feature weekend_activity_ratio criada.")

In [0]:
final_features_df = (
    
    sessions_df
    
    .join(bet_df, "user_id", "left")
    
    .join(deposit_df, "user_id", "left")
    
    .join(bonus_df, "user_id", "left")
    
    .join(engagement_df, "user_id", "left")
    
    .join(recency_df, "user_id", "left")
    
    .join(weekend_df, "user_id", "left")
)

logger.info("Tabela final de features criada.")

In [0]:
final_features_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fiap_analytics.silver.rw_db_features_churn")

# logger.info("Feature engineering finalizado com sucesso.")
print("Feature engineering finalizado com sucesso.")

In [0]:
%sql
SELECT * FROM fiap_analytics.silver.rw_db_features_churn

# Conclusão

Nesta etapa transformamos dados brutos de sessões em variáveis mais inteligentes para previsão de churn.

As features criadas representam:
- frequência,
- recência,
- comportamento financeiro,
- engajamento,
- uso de bônus.